import os, warnings, torch

import scanpy as sc
import pandas as pd

from datasets.data_manager_breast_cancer import Breast_cancer
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


In [ ]:
import os, warnings, torch

import scanpy as sc
import pandas as pd

from datasets.data_manager_breast_cancer import Breast_cancer
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


### Load dataset

In [ ]:
adata_path = '/home/wzk/ST_code/NicheTrans/2024_NicheTrans_upload_data/2023_nc_10x_breast_cancer/HBC_rep1_cell_nucleus_3channel_strength_mean.h5ad'
coordinate_path = '/home/wzk/ST_code/NicheTrans/2024_NicheTrans_upload_data/2023_nc_10x_breast_cancer/cells.csv.gz'

adata = sc.read_h5ad(adata_path)
coordinates = pd.read_csv(coordinate_path, compression='gzip')

adata.obs['x'], adata.obs['y'] = coordinates['x_centroid'].values, coordinates['y_centroid'].values

adata.obsm['spatial'] = adata.obs[['x', 'y']].values

In [ ]:
centra = adata.obs['x'].values.max()//2
testing_adata = adata[ adata.obs['x'].values >= centra ]

### Load args

In [ ]:
%run ./args/args_breast_cancer.py
args = args

### Create dataloader

In [ ]:
# create the dataset
dataset = Breast_cancer(
    adata_path=args.adata_path,
    coordinate_path=args.coordinate_path,
    ct_path=args.ct_path,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Model initialization

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=args.use_cell_type).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_breast_cancer_last.pth', device=device)


### Model inference 

In [ ]:
pd_value, gt_value, _ = collect_predictions(
    model,
    dataset.testing,
    split='test',
    device=device,
)

pd_value = pd_value.numpy()
gt_value = gt_value.numpy()


### Model evaluation

In [ ]:
from utils.evaluation import evaluator
pearson_sample_list, spearman_sample_list, _ = evaluator(pd_value, gt_value)

print(pearson_sample_list.mean())
print(spearman_sample_list.mean())

In [ ]:
pd_value = np.exp(pd_value) - 1
gt_value = np.exp(gt_value) - 1

In [ ]:
testing_adata.obs['pd_CD20'] = pd_value[:, 0]
testing_adata.obs['pd_HER2'] = pd_value[:, 1]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
sc.pl.embedding(testing_adata, basis='spatial', color='cell_CD20_mean', title=f'Ground Truth CD20', ax=axs[0], show=False, cmap=Tropic_7.mpl_colormap, size=1) 
sc.pl.embedding(testing_adata, basis='spatial', color='pd_CD20', title=f'Prediction CD20', ax=axs[1], show=False, cmap=Tropic_7.mpl_colormap, size=1) 

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
sc.pl.embedding(testing_adata, basis='spatial', color='cell_HER2_mean', title=f'Ground Truth HER2', ax=axs[0], show=False, cmap=Tropic_7.mpl_colormap, size=1) 
sc.pl.embedding(testing_adata, basis='spatial', color='pd_HER2', title=f'Prediction HER2', ax=axs[1], show=False, cmap=Tropic_7.mpl_colormap, size=1) 